# 🧬 Dark Manifold V28.1: Stable Cell Cycle Simulation

**Fixed version with proper numerical stability**

Key fixes:
- Normalized state inputs (log-transform for concentrations)
- Bounded update magnitudes with tanh
- Smaller learning rate and longer warmup
- Teacher forcing during early training

In [ ]:
#@title 1️⃣ Setup
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DOUBLING_TIME = 20 * 60  # 20 min in seconds
DT = 10.0
N_STEPS = int(DOUBLING_TIME / DT)  # 120 steps

N_EPOCHS = 2000
BATCH_SIZE = 32
LEARNING_RATE = 5e-4  # Lower LR

print(f'Simulating {DOUBLING_TIME/60:.0f} min in {N_STEPS} steps')

In [ ]:
#@title 2️⃣ Ground Truth (same as before)

class EcoliCellCycle:
    def __init__(self):
        self.metabolites = {
            'ATP': 3.65, 'ADP': 0.22, 'AMP': 0.06,
            'NAD': 2.6, 'NADH': 0.08, 'NADP': 0.002, 'NADPH': 0.12,
            'Glucose_ext': 40.0, 'G6P': 0.1, 'F6P': 0.06,
            'FBP': 0.02, 'G3P': 0.01, 'PEP': 0.04,
            'Pyruvate': 3.4, 'Acetyl_CoA': 0.6,
            'Citrate': 0.4, 'AKG': 0.3, 'Succinate': 0.5,
            'Malate': 0.2, 'OAA': 0.02,
            'Glutamate': 100.0, 'Glutamine': 4.0,
            'Phosphate': 17.8, 'CO2': 0.02,
        }
        self.met_names = list(self.metabolites.keys())
        self.n_met = len(self.met_names)
        self.met_idx = {m: i for i, m in enumerate(self.met_names)}
        self.cell_vars = ['Volume', 'DNA_replicated', 'Division_progress']
        self.n_cell = len(self.cell_vars)
        self.pathways = ['glucose_uptake', 'glycolysis', 'tca_cycle',
                         'atp_synthase', 'dna_replication', 'protein_synthesis',
                         'lipid_synthesis', 'cell_wall_synthesis']
        self.n_pathways = len(self.pathways)
        self.n_state = self.n_met + self.n_cell
        
    def get_initial_state(self, batch_size, variation=0.1):
        M0 = np.array([self.metabolites[m] for m in self.met_names])
        M = np.tile(M0, (batch_size, 1))
        M *= np.exp(np.random.randn(batch_size, self.n_met) * variation)
        C = np.zeros((batch_size, self.n_cell))
        C[:, 0] = 1.0
        state = np.concatenate([M, C], axis=1)
        return torch.tensor(state, dtype=torch.float32)
    
    def compute_phase(self, t, T=DOUBLING_TIME):
        frac = t / T
        if frac < 0.25: return 'G1', 0
        elif frac < 0.65: return 'S', 1
        elif frac < 0.85: return 'G2', 2
        else: return 'M', 3
    
    def step(self, state, t, dt=DT):
        batch_size = state.shape[0]
        M = state[:, :self.n_met].numpy().copy()
        C = state[:, self.n_met:].numpy().copy()
        phase, _ = self.compute_phase(t)
        
        atp = np.clip(M[:, self.met_idx['ATP']], 0.01, 100)
        adp = np.clip(M[:, self.met_idx['ADP']], 0.01, 100)
        glucose = np.clip(M[:, self.met_idx['Glucose_ext']], 0.01, 100)
        pyr = np.clip(M[:, self.met_idx['Pyruvate']], 0.01, 100)
        energy_charge = atp / (atp + adp + 0.01)
        
        fluxes = np.zeros((batch_size, self.n_pathways))
        v_glucose = 0.5 * glucose / (0.1 + glucose)
        fluxes[:, 0] = v_glucose
        v_glyc = 0.8 * v_glucose * (1 - 0.3 * energy_charge)
        fluxes[:, 1] = v_glyc
        v_tca = 0.6 * pyr / (0.5 + pyr) * energy_charge
        fluxes[:, 2] = v_tca
        v_atp_syn = 0.9 * (1.0 - energy_charge) * (v_glyc + v_tca)
        fluxes[:, 3] = v_atp_syn
        v_dna = 0.8 * energy_charge if phase == 'S' else 0.05
        fluxes[:, 4] = v_dna
        v_prot = 0.7 * energy_charge * (1.2 if phase in ['G1', 'G2'] else 0.8)
        fluxes[:, 5] = v_prot
        v_lipid = 0.4 * energy_charge * np.clip(C[:, 0], 0.1, 10)
        fluxes[:, 6] = v_lipid
        v_wall = 0.3 * energy_charge
        fluxes[:, 7] = v_wall
        
        # Smaller, bounded updates
        scale = dt / 100  # Reduce update magnitude
        dM = np.zeros_like(M)
        dM[:, self.met_idx['Glucose_ext']] = -v_glucose * scale
        dM[:, self.met_idx['G6P']] = (v_glucose - v_glyc) * scale
        dM[:, self.met_idx['Pyruvate']] = (2 * v_glyc - v_tca) * scale
        dM[:, self.met_idx['ATP']] = (2 * v_glyc + v_atp_syn - v_prot * 2 - v_dna) * scale
        dM[:, self.met_idx['ADP']] = -dM[:, self.met_idx['ATP']] * 0.5
        dM[:, self.met_idx['NADH']] = (v_glyc + v_tca - v_atp_syn) * 0.05 * scale
        dM += np.random.randn(*dM.shape) * 0.001
        M_new = np.clip(M + dM, 1e-4, 500.0)
        
        dC = np.zeros_like(C)
        growth_rate = np.log(2) / DOUBLING_TIME
        dC[:, 0] = C[:, 0] * growth_rate * dt * np.clip(energy_charge, 0.5, 1.0)
        if phase == 'S':
            dC[:, 1] = 0.01 * energy_charge
        if phase == 'M':
            dC[:, 2] = 0.02
        C_new = np.clip(C + dC, 0, 5)
        
        next_state = np.concatenate([M_new, C_new], axis=1)
        return torch.tensor(next_state, dtype=torch.float32), torch.tensor(fluxes, dtype=torch.float32)
    
    def generate_cycle(self, batch_size=1):
        state = self.get_initial_state(batch_size)
        states = [state]
        fluxes_list = []
        phases = []
        for step in range(N_STEPS):
            t = step * DT
            state, fluxes = self.step(state, t)
            states.append(state)
            fluxes_list.append(fluxes)
            phases.append(self.compute_phase(t)[0])
        return {
            'states': torch.stack(states, dim=1),
            'fluxes': torch.stack(fluxes_list, dim=1),
            'phases': phases,
        }

ecoli = EcoliCellCycle()
test = ecoli.generate_cycle(2)
print(f'State: {ecoli.n_state}, Pathways: {ecoli.n_pathways}')
print(f'Trajectory: {test["states"].shape}')
# Verify no collapse
print(f'ATP range: {test["states"][0, :, ecoli.met_idx["ATP"]].min():.2f} - {test["states"][0, :, ecoli.met_idx["ATP"]].max():.2f}')
print(f'Volume range: {test["states"][0, :, ecoli.n_met].min():.2f} - {test["states"][0, :, ecoli.n_met].max():.2f}')

In [ ]:
#@title 3️⃣ STABLE Cell Cycle Network

class StableCellCycleNet(nn.Module):
    '''Numerically stable cell cycle model.
    
    Key stability features:
    1. Log-transform inputs (concentrations vary over orders of magnitude)
    2. Tanh-bounded outputs (updates can't explode)
    3. Small learned scale factors
    4. Residual connection preserves state
    '''
    
    def __init__(self, n_met, n_cell, n_pathways, hidden_dim=128):
        super().__init__()
        self.n_met = n_met
        self.n_cell = n_cell
        self.n_state = n_met + n_cell
        self.n_pathways = n_pathways
        
        # Flux network with bounded output
        self.flux_net = nn.Sequential(
            nn.Linear(self.n_state, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_pathways),
            nn.Sigmoid()  # Bounded 0-1
        )
        
        # State update with bounded output
        self.state_net = nn.Sequential(
            nn.Linear(self.n_state + n_pathways, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, self.n_state),
            nn.Tanh()  # Bounded -1 to 1
        )
        
        # Learned scale for updates (initialized small)
        self.update_scale = nn.Parameter(torch.ones(self.n_state) * 0.01)
        
        # State normalization stats (will be set from data)
        self.register_buffer('state_mean', torch.zeros(self.n_state))
        self.register_buffer('state_std', torch.ones(self.n_state))
    
    def set_normalization(self, states):
        '''Set normalization from training data.'''
        self.state_mean = states.mean(dim=(0, 1))
        self.state_std = states.std(dim=(0, 1)) + 1e-6
    
    def normalize(self, state):
        '''Normalize state for network input.'''
        return (state - self.state_mean) / self.state_std
    
    def step(self, state):
        '''Single timestep with stability guarantees.'''
        # Normalize input
        state_norm = self.normalize(state)
        
        # Predict fluxes (bounded 0-1)
        fluxes = self.flux_net(state_norm)
        
        # Predict state update (bounded by tanh)
        combined = torch.cat([state_norm, fluxes], dim=-1)
        delta_norm = self.state_net(combined)  # -1 to 1
        
        # Scale update (small by design)
        delta = delta_norm * self.update_scale * self.state_std
        
        # Residual update
        next_state = state + delta
        
        # Clamp to valid ranges
        next_state = torch.clamp(next_state, min=1e-4, max=500.0)
        
        return next_state, fluxes
    
    def forward(self, initial_state, n_steps=N_STEPS):
        state = initial_state
        states = [state]
        fluxes_list = []
        
        for _ in range(n_steps):
            state, fluxes = self.step(state)
            states.append(state)
            fluxes_list.append(fluxes)
        
        return torch.stack(states, dim=1), torch.stack(fluxes_list, dim=1)

model = StableCellCycleNet(ecoli.n_met, ecoli.n_cell, ecoli.n_pathways, hidden_dim=128).to(device)

# Set normalization from sample data
sample_data = ecoli.generate_cycle(64)
model.set_normalization(sample_data['states'].to(device))

n_params = sum(p.numel() for p in model.parameters())
print(f'StableCellCycleNet: {n_params:,} parameters')
print(f'Update scale init: {model.update_scale.mean().item():.4f}')

In [ ]:
#@title 4️⃣ Training with Teacher Forcing

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=500, T_mult=2)

train_losses = []
val_corrs = []
best_corr = 0
best_state = None

print('Training with teacher forcing...')
for epoch in range(N_EPOCHS):
    model.train()
    
    # Generate ground truth
    gt_data = ecoli.generate_cycle(batch_size=BATCH_SIZE)
    gt_states = gt_data['states'].to(device)
    gt_fluxes = gt_data['fluxes'].to(device)
    
    # Teacher forcing ratio (starts high, decreases)
    tf_ratio = max(0.0, 1.0 - epoch / 1000)
    
    # Forward with teacher forcing
    state = gt_states[:, 0, :]
    states = [state]
    fluxes_list = []
    
    for t in range(N_STEPS):
        next_state, fluxes = model.step(state)
        fluxes_list.append(fluxes)
        
        # Teacher forcing: sometimes use ground truth
        if np.random.random() < tf_ratio and t < N_STEPS - 1:
            state = gt_states[:, t + 1, :]
        else:
            state = next_state
        states.append(state)
    
    pred_states = torch.stack(states, dim=1)
    pred_fluxes = torch.stack(fluxes_list, dim=1)
    
    # Loss
    state_loss = nn.functional.mse_loss(pred_states, gt_states)
    flux_loss = nn.functional.mse_loss(pred_fluxes, gt_fluxes)
    
    # Regularization: penalize large update scales
    scale_reg = 0.01 * (model.update_scale ** 2).mean()
    
    loss = state_loss + 0.1 * flux_loss + scale_reg
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
    optimizer.step()
    scheduler.step()
    
    train_losses.append(loss.item())
    
    if epoch % 200 == 0 or epoch == N_EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            val_data = ecoli.generate_cycle(batch_size=64)
            val_states = val_data['states'].to(device)
            
            # Full autoregressive (no teacher forcing)
            pred_val, _ = model(val_states[:, 0, :], N_STEPS)
            
            pred_flat = pred_val.cpu().numpy().flatten()
            true_flat = val_states.cpu().numpy().flatten()
            r, _ = pearsonr(pred_flat, true_flat)
            val_corrs.append((epoch, r))
            
            # Check for collapse
            atp_pred = pred_val[0, :, ecoli.met_idx['ATP']].cpu().numpy()
            collapsed = atp_pred.min() < 0.01
            
            if r > best_corr and not collapsed:
                best_corr = r
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
            status = '❌ COLLAPSED' if collapsed else '✅'
            print(f'Epoch {epoch:4d} | Loss: {loss.item():.4f} | Val r: {r:.4f} | TF: {tf_ratio:.2f} | {status}')

print(f'\nBest correlation: {best_corr:.4f}')

In [ ]:
#@title 5️⃣ Generate Virtual Cell

if best_state is not None:
    model.load_state_dict(best_state)
model.eval()

print('Generating virtual cell...')
with torch.no_grad():
    initial = ecoli.get_initial_state(1).to(device)
    pred_states, pred_fluxes = model(initial, N_STEPS)

states = pred_states[0].cpu().numpy()
fluxes = pred_fluxes[0].cpu().numpy()
time_seconds = (np.arange(N_STEPS + 1) * DT).tolist()

# Check for collapse
atp = states[:, ecoli.met_idx['ATP']]
print(f'ATP: {atp[0]:.2f} -> {atp[N_STEPS//2]:.2f} -> {atp[-1]:.2f} mM')
print(f'Volume: {states[0, ecoli.n_met]:.2f} -> {states[-1, ecoli.n_met]:.2f}')

metabolites = {name: states[:, i].tolist() for i, name in enumerate(ecoli.met_names)}
cell_properties = {
    'volume': states[:, ecoli.n_met].tolist(),
    'dna_replicated': states[:, ecoli.n_met + 1].tolist(),
    'division_progress': states[:, ecoli.n_met + 2].tolist(),
}
pathway_fluxes = {name: fluxes[:, i].tolist() for i, name in enumerate(ecoli.pathways)}
phases = [ecoli.compute_phase(t)[0] for t in time_seconds]

virtual_cell = {
    'metadata': {
        'organism': 'E. coli',
        'doubling_time_minutes': DOUBLING_TIME / 60,
        'model_version': 'V28.1-stable',
    },
    'time_seconds': time_seconds,
    'metabolites': metabolites,
    'cell_properties': cell_properties,
    'pathway_fluxes': pathway_fluxes,
    'phases': phases,
}

with open('virtual_cell_v28_1.json', 'w') as f:
    json.dump(virtual_cell, f, indent=2)
print('Saved virtual_cell_v28_1.json')

In [ ]:
#@title 6️⃣ Visualization

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
time_min = np.array(time_seconds) / 60

ax = axes[0, 0]
for met in ['ATP', 'ADP', 'Pyruvate', 'Glucose_ext']:
    ax.plot(time_min, metabolites[met], label=met)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Key Metabolites')
ax.legend()

ax = axes[0, 1]
ax.plot(time_min, cell_properties['volume'], 'g-', label='Volume', linewidth=2)
ax.plot(time_min, cell_properties['dna_replicated'], 'b-', label='DNA')
ax.plot(time_min, cell_properties['division_progress'], 'r-', label='Division')
ax.set_xlabel('Time (min)')
ax.set_title('Cell Cycle Progression')
ax.legend()

ax = axes[1, 0]
for p in ['glycolysis', 'tca_cycle', 'atp_synthase']:
    ax.plot(time_min[:-1], pathway_fluxes[p], label=p)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Flux')
ax.set_title('Metabolic Pathways')
ax.legend()

ax = axes[1, 1]
for p in ['dna_replication', 'protein_synthesis', 'lipid_synthesis']:
    ax.plot(time_min[:-1], pathway_fluxes[p], label=p)
ax.set_xlabel('Time (min)')
ax.set_title('Biosynthesis')
ax.legend()

ax = axes[2, 0]
atp = np.array(metabolites['ATP'])
adp = np.array(metabolites['ADP'])
amp = np.array(metabolites['AMP'])
ec = (atp + 0.5 * adp) / (atp + adp + amp + 0.01)
ax.plot(time_min, ec, 'purple', linewidth=2)
ax.axhline(0.85, color='green', linestyle='--', label='Optimal')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Energy Charge')
ax.set_title('Cellular Energy')
ax.legend()

ax = axes[2, 1]
ax.semilogy(train_losses)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Training (best r={best_corr:.4f})')

plt.tight_layout()
plt.savefig('v28_1_results.png', dpi=150)
plt.show()

In [ ]:
#@title 7️⃣ Download
torch.save({
    'model_state_dict': best_state or model.state_dict(),
    'best_corr': best_corr,
    'config': {'n_met': ecoli.n_met, 'n_cell': ecoli.n_cell, 'n_pathways': ecoli.n_pathways}
}, 'dark_manifold_v28_1.pt')

from google.colab import files
files.download('dark_manifold_v28_1.pt')
files.download('virtual_cell_v28_1.json')
files.download('v28_1_results.png')